In [15]:
from dotenv import load_dotenv

load_dotenv()

True

In [16]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable
from langchain.chat_models import init_chat_model

large_model  = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")

standard_model = init_chat_model("qwen/qwen3-32b", model_provider="groq")


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [17]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[state_based_model],
    system_prompt="What is dog"
)

In [18]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

<think>
Okay, the user is asking if I watered the office plant today. Since I'm an AI, I can't physically do that. I need to explain that I don't have a physical form and can't interact with the real world. But maybe they want to know if someone else did it. I should ask if they're referring to a specific person or system. Also, I should check if there's a way to track such tasks, like a calendar or a reminder app. I need to make sure my response is helpful and clear, guiding them to provide more details so I can assist better.
</think>

I'm Qwen, a language model developed by Tongyi Lab. I don't have a physical body and can't interact with the real world directly, so I'm unable to water plants or perform any physical tasks. If you're asking about the care of a specific plant, you might want to check with the person responsible for office maintenance or see if there's an automated watering system in place. Let me know if you'd like help with something else!


In [19]:
print(response["messages"][-1].response_metadata["model_name"])

qwen/qwen3-32b


In [20]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

As a digital AI assistant, I don't have a physical presence, so I don't have an office plant to water, fertilize, or replace. I exist solely to provide information and assist with text-based conversations. If you have any questions about plant care or would like advice on how to care for a specific plant, I'd be happy to help!


In [21]:
print(response["messages"][-1].response_metadata["model_name"])

llama-3.3-70b-versatile
